# 01 – Load & Clean: Greek Super League matches and betting odds (2016/17–2025/26)
**Source:** football-data.co.uk (Greece, file G1.csv per season). Column dictionary: https://www.football-data.co.uk/notes.txt

**Goal of this notebook**
1. Load the 10 season files and stack them in one table (1 row = 1 match).
2. Check which odds columns exist in each season.
3. Clean: empty rows, dates, results, missing odds, duplicates.
4. Create new columns: implied probabilities, bookmaker margin (overround), total goals.
5. Export a clean CSV and a SQLite database for the SQL and Power BI steps.

In [1]:
import glob
import os
import sqlite3

import numpy as np
import pandas as pd

BASE = r"C:\Projects\greek-betting-markets"
RAW = os.path.join(BASE, "data-raw")
CLEAN = os.path.join(BASE, "data-clean")
os.makedirs(CLEAN, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Load the 10 files
The season is in the file name: `G1_1617.csv` → `2016/17`. Some files have more columns than others, so we first look at each one separately.

In [2]:
files = sorted(glob.glob(os.path.join(RAW, "G1_*.csv")))
print(len(files), "files")

raw = {}
for f in files:
    code = os.path.basename(f)[3:7]                 # '1617'
    season = f"20{code[:2]}/{code[2:]}"             # '2016/17'
    df = pd.read_csv(f, encoding="latin-1")
    df = df.dropna(how="all")                       # fully empty rows at the end of the file
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # empty trailing columns
    raw[season] = df
    print(f"{season}: {len(df):>4} rows, {df.shape[1]:>3} columns")

10 files
2016/17:  240 rows,  52 columns
2017/18:  240 rows,  64 columns
2018/19:  240 rows,  61 columns
2019/20:  240 rows, 105 columns
2020/21:  240 rows, 105 columns
2021/22:  240 rows, 105 columns
2022/23:  240 rows, 105 columns
2023/24:  240 rows, 105 columns
2024/25:  233 rows, 119 columns
2025/26:  236 rows, 131 columns


## 2. Which odds columns exist in each season?
- `B365H/D/A` = Bet365 odds for Home / Draw / Away (1X2 market)
- `PSH/D/A` = Pinnacle odds (a low-margin, "sharp" bookmaker – our benchmark)
- `B365>2.5`, `B365<2.5` = Bet365 Over / Under 2.5 goals
- `P>2.5`, `P<2.5` = Pinnacle Over / Under 2.5 goals
- columns ending in `C` (e.g. `B365CH`) = closing odds, just before kick-off

In [3]:
check_cols = ["Time", "B365H", "PSH", "B365>2.5", "P>2.5", "Avg>2.5", "BbAv>2.5", "B365CH", "PSCH"]
availability = pd.DataFrame(
    {season: [c in df.columns for c in check_cols] for season, df in raw.items()},
    index=check_cols,
).T
availability

,Time,B365H,PSH,B365>2.5,P>2.5,Avg>2.5,BbAv>2.5,B365CH,PSCH
2016/17,False,True,True,False,False,False,True,False,True
2017/18,False,True,True,False,False,False,True,False,True
2018/19,False,True,True,False,False,False,True,False,True
2019/20,True,True,True,True,True,True,False,True,True
2020/21,True,True,True,True,True,True,False,True,True
2021/22,True,True,True,True,True,True,False,True,True
2022/23,True,True,True,True,True,True,False,True,True
2023/24,True,True,True,True,True,True,False,True,True
2024/25,True,True,True,True,True,True,False,True,True
2025/26,True,True,True,True,True,True,False,True,True


## 3. Stack the seasons into one table
We keep only the columns we need. `reindex` creates any missing column as empty (NaN), so all seasons get the same structure.

In [4]:
keep = ["Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "B365H", "B365D", "B365A", "PSH", "PSD", "PSA",
        "B365>2.5", "B365<2.5", "P>2.5", "P<2.5"]

parts = []
for season, df in raw.items():
    part = df.reindex(columns=keep).copy()
    part.insert(0, "Season", season)
    parts.append(part)

matches = pd.concat(parts, ignore_index=True)
matches = matches.rename(columns={"B365>2.5": "B365_Over25", "B365<2.5": "B365_Under25",
                                  "P>2.5": "PS_Over25", "P<2.5": "PS_Under25"})
print(matches.shape)
matches.head()

(2389, 19)


,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A,PSH,PSD,PSA,B365_Over25,B365_Under25,PS_Over25,PS_Under25
0,2016/17,G1,10/09/16,NaN,Kerkyra,Platanias,0.0,1.0,A,2.40,3.00,3.20,2.54,3.10,3.25,NaN,NaN,NaN,NaN
1,2016/17,G1,10/09/16,NaN,Levadeiakos,Panathinaikos,0.0,3.0,A,6.50,3.40,1.62,6.43,3.84,1.63,NaN,NaN,NaN,NaN
2,2016/17,G1,10/09/16,NaN,Panionios,Asteras Tripolis,3.0,0.0,H,2.50,3.00,3.00,2.76,3.13,2.93,NaN,NaN,NaN,NaN
3,2016/17,G1,11/09/16,NaN,AEK,Xanthi,4.0,1.0,H,1.36,4.33,9.50,1.43,4.31,10.94,NaN,NaN,NaN,NaN
4,2016/17,G1,11/09/16,NaN,Giannina,Atromitos,3.0,0.0,H,2.40,3.00,3.20,2.44,3.11,3.40,NaN,NaN,NaN,NaN


## 4. Cleaning checks
### 4a. Rows without teams or result (e.g. empty lines, postponed matches)

In [5]:
no_teams = matches["HomeTeam"].isna() | matches["AwayTeam"].isna()
no_result = matches["FTR"].isna()
print("Rows without teams :", no_teams.sum())
print("Rows without result:", no_result.sum())
matches = matches[~no_teams & ~no_result].copy()
print("Rows after removal :", len(matches))

Rows without teams : 0
Rows without result: 1
Rows after removal : 2388


### 4b. Dates
Older files write the year with 2 digits (`23/08/17`), newer with 4 (`23/08/2025`). `dayfirst=True` because the format is day/month/year.

In [6]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")
print("Missing dates:", matches["Date"].isna().sum())
matches.groupby("Season")["Date"].agg(["min", "max", "count"])

Missing dates: 0


,min,max,count
Season,,,
2016/17,2016-09-10,2017-04-30,240
2017/18,2017-08-19,2018-05-07,240
2018/19,2018-08-25,2019-05-05,239
2019/20,2019-08-24,2020-07-19,240
2020/21,2020-09-11,2021-05-19,240
2021/22,2021-09-11,2022-05-17,240
2022/23,2022-08-19,2023-05-14,240
2023/24,2023-08-18,2024-05-19,240
2024/25,2024-08-17,2025-05-18,233


### 4c. Results, goals and duplicates
`FTR` must be only H, D or A, and it must agree with the goals (`FTHG` home goals, `FTAG` away goals).

In [7]:
print(matches["FTR"].value_counts())

matches["FTHG"] = matches["FTHG"].astype(int)
matches["FTAG"] = matches["FTAG"].astype(int)
expected = np.select([matches["FTHG"] > matches["FTAG"], matches["FTHG"] < matches["FTAG"]], ["H", "A"], default="D")
print("FTR that does not match the goals:", (matches["FTR"] != expected).sum())

dups = matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()
print("Duplicate matches:", dups)

FTR
H    1036
A     706
D     646
Name: count, dtype: int64
FTR that does not match the goals: 0
Duplicate matches: 0


### 4d. Team names
The same club must have one spelling in all seasons, otherwise SQL and Power BI will count it as two teams.

In [8]:
teams = pd.concat([matches["HomeTeam"], matches["AwayTeam"]]).str.strip()
team_seasons = (pd.concat([matches[["Season", "HomeTeam"]].rename(columns={"HomeTeam": "Team"}),
                           matches[["Season", "AwayTeam"]].rename(columns={"AwayTeam": "Team"})])
                .assign(Team=lambda d: d["Team"].str.strip())
                .groupby("Team")["Season"].nunique().sort_values(ascending=False))
print(len(team_seasons), "different team names")
team_seasons

25 different team names


Team
AEK                 10
Asteras Tripolis    10
Atromitos           10
Panetolikos         10
Panathinaikos       10
PAOK                10
Olympiakos          10
Aris                 8
Lamia                8
OFI Crete            8
Volos NFC            7
Giannina             7
Levadeiakos          6
Larisa               6
Apollon              4
Panionios            4
Xanthi               4
Panserraikos         3
Kifisia              2
Kerkyra              2
Ionikos              2
Platanias            2
Iraklis              1
Veria                1
Athens Kallithea     1
Name: Season, dtype: int64

### 4e. Missing odds per season
A match with missing odds stays in the table (it counts for results and home advantage), but it is left out of the margin calculations for that bookmaker.

In [9]:
odds_cols = ["B365H", "B365D", "B365A", "PSH", "PSD", "PSA", "B365_Over25", "B365_Under25", "PS_Over25", "PS_Under25"]
for c in odds_cols:
    matches[c] = pd.to_numeric(matches[c], errors="coerce")

matches[odds_cols].isna().groupby(matches["Season"]).sum()

,B365H,B365D,B365A,PSH,PSD,PSA,B365_Over25,B365_Under25,PS_Over25,PS_Under25
Season,,,,,,,,,,
2016/17,3,3,3,49,49,49,240,240,240,240
2017/18,10,10,10,60,60,60,240,240,240,240
2018/19,8,8,8,31,31,31,239,239,239,239
2019/20,3,3,3,20,20,20,3,3,20,20
2020/21,4,4,4,17,17,17,4,4,17,17
2021/22,2,2,2,0,0,0,3,3,0,0
2022/23,0,0,0,1,1,1,0,0,1,1
2023/24,0,0,0,0,0,0,0,0,0,0
2024/25,0,0,0,0,0,0,0,0,0,0


In [10]:
# Odds must be above 1.00. Anything else is a data error.
bad_odds = (matches[odds_cols] <= 1).sum()
print(bad_odds[bad_odds > 0] if bad_odds.sum() else "No odds <= 1.00")
print()
matches[odds_cols].describe().T[["count", "min", "50%", "max"]]

No odds <= 1.00



,count,min,50%,max
B365H,2350.0,1.02,2.150,34.00
B365D,2350.0,1.50,3.400,13.00
B365A,2350.0,1.07,3.500,41.00
PSH,2028.0,1.03,2.190,19.94
PSD,2028.0,2.16,3.400,21.57
PSA,2028.0,1.16,3.635,40.30
B365_Over25,1651.0,1.25,2.100,3.10
B365_Under25,1651.0,1.36,1.700,4.00
PS_Over25,1449.0,1.21,2.160,3.16
PS_Under25,1449.0,1.34,1.690,4.04


## 5. New columns
**Implied probability** = 1 / decimal odds. Example: odds 2.00 → 50%.

**Overround (bookmaker margin)** = sum of the implied probabilities of all outcomes − 1. In a fair market the probabilities add up to 100%. The bookmaker shortens the odds, so the sum is above 100%. The extra part is the margin.

**Fair (normalised) probability** = implied probability / sum, so the three outcomes add up to exactly 100%. We use it to check if the odds are accurate.

In [11]:
for bk in ["B365", "PS"]:
    inv = 1 / matches[[f"{bk}H", f"{bk}D", f"{bk}A"]]
    total = inv.sum(axis=1, min_count=3)                 # NaN if any of the 3 odds is missing
    matches[f"{bk}_Overround_1X2"] = total - 1
    for o in ["H", "D", "A"]:
        matches[f"{bk}_Prob{o}"] = (1 / matches[f"{bk}{o}"]) / total

    ou = (1 / matches[[f"{bk}_Over25", f"{bk}_Under25"]]).sum(axis=1, min_count=2)
    matches[f"{bk}_Overround_OU25"] = ou - 1
    matches[f"{bk}_ProbOver25"] = (1 / matches[f"{bk}_Over25"]) / ou

matches["TotalGoals"] = matches["FTHG"] + matches["FTAG"]
matches["Over25"] = (matches["TotalGoals"] > 2.5).astype(int)
matches["HomeWin"] = (matches["FTR"] == "H").astype(int)
matches["Draw"] = (matches["FTR"] == "D").astype(int)
matches["AwayWin"] = (matches["FTR"] == "A").astype(int)

# Which outcome the Bet365 odds made favourite (lowest odds)
odds3 = matches[["B365H", "B365D", "B365A"]]
has_odds = odds3.notna().all(axis=1)
matches["B365_Favourite"] = pd.Series(None, index=matches.index, dtype="object")
matches.loc[has_odds, "B365_Favourite"] = odds3[has_odds].idxmin(axis=1).str[-1]
matches["FavouriteWon"] = np.where(matches["B365_Favourite"].isna(), np.nan,
                                   (matches["B365_Favourite"] == matches["FTR"]).astype(float))

matches[["Season", "HomeTeam", "AwayTeam", "FTR", "B365H", "B365D", "B365A",
         "B365_Overround_1X2", "B365_ProbH", "PS_Overround_1X2"]].head()

,Season,HomeTeam,AwayTeam,FTR,B365H,B365D,B365A,B365_Overround_1X2,B365_ProbH,PS_Overround_1X2
0,2016/17,Kerkyra,Platanias,A,2.40,3.00,3.20,0.062500,0.392157,0.023974
1,2016/17,Levadeiakos,Panathinaikos,A,6.50,3.40,1.62,0.065248,0.144423,0.029435
2,2016/17,Panionios,Asteras Tripolis,H,2.50,3.00,3.00,0.066667,0.375000,0.023105
3,2016/17,AEK,Xanthi,H,1.36,4.33,9.50,0.071504,0.686226,0.022727
4,2016/17,Giannina,Atromitos,H,2.40,3.00,3.20,0.062500,0.392157,0.025497


## 6. First look: one row per season

In [12]:
season_summary = matches.groupby("Season").agg(
    Matches=("FTR", "size"),
    HomeWinPct=("HomeWin", "mean"),
    DrawPct=("Draw", "mean"),
    AwayWinPct=("AwayWin", "mean"),
    AvgGoals=("TotalGoals", "mean"),
    Over25Pct=("Over25", "mean"),
    B365_Margin_1X2=("B365_Overround_1X2", "mean"),
    PS_Margin_1X2=("PS_Overround_1X2", "mean"),
    B365_Margin_OU25=("B365_Overround_OU25", "mean"),
    PS_Margin_OU25=("PS_Overround_OU25", "mean"),
)
pct_cols = [c for c in season_summary.columns if c not in ("Matches", "AvgGoals")]
season_summary.style.format({**{c: "{:.1%}" for c in pct_cols}, "AvgGoals": "{:.2f}"})

,Matches,HomeWinPct,DrawPct,AwayWinPct,AvgGoals,Over25Pct,B365_Margin_1X2,PS_Margin_1X2,B365_Margin_OU25,PS_Margin_OU25
Season,,,,,,,,,,
2016/17,240,48.3%,27.5%,24.2%,2.32,40.4%,7.3%,2.5%,nan%,nan%
2017/18,240,45.4%,30.0%,24.6%,2.21,37.5%,7.1%,2.7%,nan%,nan%
2018/19,239,48.5%,23.0%,28.5%,2.28,41.4%,6.8%,5.3%,nan%,nan%
2019/20,240,44.6%,29.2%,26.2%,2.36,44.6%,7.1%,5.4%,6.3%,6.3%
2020/21,240,37.9%,29.2%,32.9%,2.20,38.3%,7.4%,5.4%,6.3%,5.3%
2021/22,240,42.5%,25.0%,32.5%,2.38,45.0%,5.7%,5.8%,6.2%,5.6%
2022/23,240,40.4%,30.0%,29.6%,2.35,40.8%,5.5%,5.4%,6.0%,5.2%
2023/24,240,44.6%,25.8%,29.6%,2.96,59.2%,5.4%,5.7%,5.6%,5.4%
2024/25,233,40.3%,22.3%,37.3%,2.44,47.2%,6.3%,6.1%,5.9%,5.2%


## 7. Export
- `data-clean/matches_clean.csv` → for checking and for the GitHub repo
- `data-clean/greek_betting.db` → SQLite database for DBeaver (table `matches`)

The date is saved as text `YYYY-MM-DD` so that SQLite and Power BI read it the same way.

In [13]:
out = matches.sort_values(["Date", "HomeTeam"]).copy()
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

csv_path = os.path.join(CLEAN, "matches_clean.csv")
db_path = os.path.join(CLEAN, "greek_betting.db")
out.to_csv(csv_path, index=False)

with sqlite3.connect(db_path) as con:
    out.to_sql("matches", con, if_exists="replace", index=False)
    n = con.execute("SELECT COUNT(*) FROM matches").fetchone()[0]

print("CSV :", csv_path, "|", len(out), "rows,", out.shape[1], "columns")
print("DB  :", db_path, "| table matches:", n, "rows")

CSV : C:\Projects\greek-betting-markets\data-clean\matches_clean.csv | 2388 rows, 38 columns
DB  : C:\Projects\greek-betting-markets\data-clean\greek_betting.db | table matches: 2388 rows


In [14]:
# Check 2023/24: is the high number of goals real or a data problem?
s = matches[matches["Season"] == "2023/24"]
print("Dates:", s["Date"].min().date(), "to", s["Date"].max().date())
print("\nMatches by total goals:")
print(s["TotalGoals"].value_counts().sort_index())

# What the bookmaker expected vs what happened, every season
check = matches.groupby("Season").agg(
    Over25_actual=("Over25", "mean"),
    Over25_expected_B365=("B365_ProbOver25", "mean"),
    AvgGoals=("TotalGoals", "mean"),
)
print(check.round(3))

# The 10 matches with the most goals in 2023/24
s.sort_values("TotalGoals", ascending=False)[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].head(10)

Dates: 2023-08-18 to 2024-05-19

Matches by total goals:
TotalGoals
0    11
1    28
2    59
3    58
4    44
5    28
6    11
8     1
Name: count, dtype: int64
         Over25_actual  Over25_expected_B365  AvgGoals
Season                                                
2016/17          0.404                   NaN     2.317
2017/18          0.375                   NaN     2.208
2018/19          0.414                   NaN     2.285
2019/20          0.446                 0.450     2.362
2020/21          0.383                 0.436     2.204
2021/22          0.450                 0.435     2.375
2022/23          0.408                 0.441     2.350
2023/24          0.592                 0.494     2.958
2024/25          0.472                 0.466     2.442
2025/26          0.496                 0.472     2.568


,Date,HomeTeam,AwayTeam,FTHG,FTAG
1702,2023-09-17,Kifisia,Panserraikos,4,4
1883,2024-04-07,Lamia,Olympiakos,1,5
1783,2023-12-21,Kifisia,PAOK,0,6
1848,2024-02-28,Asteras Tripolis,Kifisia,3,3
1823,2024-02-04,AEK,Asteras Tripolis,4,2
1809,2024-01-21,Volos NFC,PAOK,1,5
1855,2024-03-03,Aris,AEK,3,3
1852,2024-02-28,AEK,Giannina,4,2
1746,2023-11-05,Olympiakos,PAOK,2,4
1761,2023-11-26,Lamia,Atromitos,3,3
